# Exercise 3.2 — Fine-Tuning CLIP with LoRA

## 0. Installazione dipendenze

In [1]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "open_clip_torch",      # open_clip: API pulita, LoRA facile da iniettare
    "datasets",
    "numpy",
    "tqdm",
    "Pillow"
])
print('✅ Dipendenze installate')

✅ Dipendenze installate


## 1. Setup e caricamento modello

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16',
    pretrained='openai',
    device=device
)

tokenizer = open_clip.get_tokenizer('ViT-B-16')

torch.set_float32_matmul_precision('high')

print(f"✅ Modello caricato: ViT-B/16 (openai/clip-vit-base-patch16)")
print(f"   CLIP logit_scale (temperature): {model.logit_scale.exp().item():.4f}")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Flash Attention disponibile: {torch.backends.cuda.flash_sdp_enabled()}")
print(f"Memory Efficient Attention: {torch.backends.cuda.mem_efficient_sdp_enabled()}")

model.eval()
dummy = torch.randn(64, 3, 224, 224, device=device)

import time
with torch.no_grad():

    for _ in range(3):
        model.encode_image(dummy)

    torch.cuda.synchronize()
    t0 = time.time()

    for _ in range(10):
        model.encode_image(dummy)

    torch.cuda.synchronize()
    elapsed = time.time() - t0

print(f"\nForward pass (batch=64): {elapsed/10*1000:.1f} ms/batch")
print(f"Throughput: {640/elapsed:.0f} img/sec")

Device: cuda


c:\Users\checc\anaconda3\envs\clip_lora\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


✅ Modello caricato: ViT-B/16 (openai/clip-vit-base-patch16)
   CLIP logit_scale (temperature): 100.0000
PyTorch: 2.7.1+cu118
CUDA disponibile: True
GPU: NVIDIA GeForce RTX 2050
Flash Attention disponibile: True
Memory Efficient Attention: True

Forward pass (batch=64): 731.9 ms/batch
Throughput: 87 img/sec


## 2. Implementazione LoRA

Iniettare LoRA **manualmente** nei layer `MultiheadAttention` dell'image encoder.
Questo approccio segue il paper CLIP-LoRA (Zanella & Ben Ayed, CVPRW 2024) ed evita
i problemi di compatibilità con la libreria PEFT su architetture CLIP. 

NON HA FUNZIONATO

## 3. Caricamento Dataset — ImageNet-Sketch

In [3]:
import json, urllib.request
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# Labels delle 1000 classi ImageNet
url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
urllib.request.urlretrieve(url, "imagenet_labels.json")
with open("imagenet_labels.json") as f:
    imagenet_class_names = json.load(f)  # lista di 1000 stringhe

print("Caricamento ImageNet-Sketch...")
full_sketch_ds = load_dataset("imagenet_sketch", split="train", trust_remote_code=True)

# Shuffle deterministico e split 80/20
full_shuffled = full_sketch_ds.shuffle(seed=42)
train_size    = int(0.8 * len(full_shuffled))
sketch_train  = full_shuffled.select(range(train_size))
sketch_val    = full_shuffled.select(range(train_size, len(full_shuffled)))

print(f"✅ Dataset pronto — Train: {len(sketch_train)} | Val: {len(sketch_val)}")
print(f"   Esempio label: {imagenet_class_names[sketch_train[0]['label']]}")

Caricamento ImageNet-Sketch...
✅ Dataset pronto — Train: 40711 | Val: 10178
   Esempio label: African rock python


## 4. Pre-elaborazione in RAM
Preprocessiamo le immagini e i prompt una volta sola e salviamo tutto in tensori,
come nella tua versione 2 (approccio corretto per velocità).

In [4]:
def build_tensor_dataset(hf_dataset, num_samples=None):
    """
    Preprocessa:
    - immagini → tensor [3,224,224]
    - label → int
    """

    n = num_samples or len(hf_dataset)

    all_pv = []
    all_labels = []

    print(f"Pre-elaborazione di {n} campioni...")

    for i in tqdm(range(n)):
        ex = hf_dataset[i]

        image = ex["image"].convert("RGB")
        label = ex["label"]

        pv = preprocess(image)

        all_pv.append(pv)
        all_labels.append(label)

    return TensorDataset(
        torch.stack(all_pv),
        torch.tensor(all_labels)
    )


# Usiamo 5000 campioni per il train (addestriamo LoRA, non il modello completo)
# e tutto il val per una valutazione robusta
TRAIN_SAMPLES = 5000

train_tensor_ds = build_tensor_dataset(sketch_train, num_samples=TRAIN_SAMPLES)
val_tensor_ds   = build_tensor_dataset(sketch_val)   # tutti i campioni val

train_loader = DataLoader(train_tensor_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_tensor_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"\n✅ DataLoader pronti")
print(f"   Train batch per epoca: {len(train_loader)}")
print(f"   Val batch: {len(val_loader)}")

Pre-elaborazione di 5000 campioni...


100%|██████████| 5000/5000 [00:48<00:00, 104.15it/s]


Pre-elaborazione di 10178 campioni...


100%|██████████| 10178/10178 [01:34<00:00, 108.05it/s]



✅ DataLoader pronti
   Train batch per epoca: 79
   Val batch: 160


## 5. Valutazione Zero-Shot (Baseline)

Valutiamo CLIP *prima* di qualsiasi fine-tuning.
Nota: usiamo tutti i 1000 prompt testuali per la classificazione zero-shot.

In [5]:
# ═══════════════════════════════════════════════════════════════════
# SEZIONE 5 — ZERO-SHOT EVALUATION (BASELINE CORRETTA)
# ═══════════════════════════════════════════════════════════════════

import torch
from tqdm import tqdm

@torch.no_grad()
def build_text_features(clip_model, class_names, prompt_template):
    """
    Precompute text embeddings UNA SOLA VOLTA
    """
    clip_model.eval()

    text_feats = []

    for start in range(0, len(class_names), 100):
        end = min(start + 100, len(class_names))

        prompts = [
            prompt_template.format(c)
            for c in class_names[start:end]
        ]

        tokens = tokenizer(prompts).to(device)

        feats = clip_model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        text_feats.append(feats)

    return torch.cat(text_feats, dim=0)


@torch.no_grad()
def evaluate_zero_shot(clip_model, loader, text_feats):
    """
    Eval standard zero-shot (NO prompt recomputation)
    """
    clip_model.eval()

    scale = clip_model.logit_scale.exp()

    correct, total = 0, 0

    for pv, labels in tqdm(loader, desc="Zero-shot eval", leave=False):
        pv = pv.to(device)
        labels = labels.to(device)

        img_feat = clip_model.encode_image(pv)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        logits = scale * img_feat @ text_feats.T
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


# ═══════════════════════════════════════════════════════════════════
# BASELINE — SINGLE PROMPT
# ═══════════════════════════════════════════════════════════════════

print("Calcolo Zero-Shot baseline...")

base_prompt = "a sketch of a {}"

text_feats_base = build_text_features(
    model,
    imagenet_class_names,
    base_prompt
)

zeroshot_acc = evaluate_zero_shot(
    model,
    val_loader,
    text_feats_base
)

print(f"\n🎯 Zero-Shot Accuracy: {zeroshot_acc * 100:.2f}%")


# ═══════════════════════════════════════════════════════════════════
# PROMPT ABALATION (IMPORTANT FIX)
# ═══════════════════════════════════════════════════════════════════

print("\nPrompt ablation study...\n")

prompt_list = [
    "a sketch of a {}",
    "a black and white sketch of a {}",
    "a hand-drawn sketch of a {}",
    "a pencil drawing of a {}",
    "a line drawing of a {}"
]

prompt_results = []

for prompt in prompt_list:

    text_feats = build_text_features(
        model,
        imagenet_class_names,
        prompt
    )

    acc = evaluate_zero_shot(
        model,
        val_loader,
        text_feats
    )

    prompt_results.append((prompt, acc))


# ═══════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════

best_prompt = None
best_acc = 0

print("\nPrompt comparison:\n")

for p, acc in prompt_results:
    print(f"{p:<45} {acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc
        best_prompt = p

print("-"*60)
print(f"Best Prompt: {best_prompt}")
print(f"Best Accuracy: {best_acc*100:.2f}%")

Calcolo Zero-Shot baseline...



🎯 Zero-Shot Accuracy: 42.64%

Prompt ablation study...




Prompt comparison:

a sketch of a {}                              42.64%
a black and white sketch of a {}              43.14%
a hand-drawn sketch of a {}                   42.97%
a pencil drawing of a {}                      42.69%
a line drawing of a {}                        43.20%
------------------------------------------------------------
Best Prompt: a line drawing of a {}
Best Accuracy: 43.20%


## 6. Iniezione LoRA e Training

Adesso iniettamo LoRA e addestriamo solo le matrici A, B su 5000 sketch.

Pre-calcolo feature

In [6]:
# ═══════════════════════════════════════════════════════════════════
# SEZIONE 6 — CLIP Adapter (PEFT Fine-Tuning + VALIDATION CORRECTED)
# ═══════════════════════════════════════════════════════════════════

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from tqdm import tqdm


# ────────────────────────────────────────────────────────────────
# 1. CLIP ADAPTER
# ────────────────────────────────────────────────────────────────

class CLIPAdapter(nn.Module):

    def __init__(self, feat_dim=512, bottleneck=64, alpha=0.6):
        super().__init__()

        self.fc1 = nn.Linear(feat_dim, bottleneck, bias=True)
        self.fc2 = nn.Linear(bottleneck, feat_dim, bias=True)

        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

        self.alpha = nn.Parameter(torch.tensor(alpha))

    def forward(self, x):
        residual = F.relu(self.fc1(x))
        residual = self.fc2(residual)

        alpha = torch.sigmoid(self.alpha)
        return x + alpha * residual


# ────────────────────────────────────────────────────────────────
# 2. TRAIN / VAL SPLIT (FEATURE-LEVEL SPLIT)
# ────────────────────────────────────────────────────────────────

full_ds = train_tensor_ds  # 5000 samples già preprocessati

train_size = int(0.9 * len(full_ds))
val_size   = len(full_ds) - train_size

train_ds, val_ds = random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)


# ────────────────────────────────────────────────────────────────
# 3. PRECOMPUTE FEATURES (TRAIN + VAL SEPARATI ✔ FIX CRITICO)
# ────────────────────────────────────────────────────────────────

print("\nPrecomputing image features...")

model.eval()

# TRAIN FEATURES
train_feats, train_labels = [], []

with torch.no_grad():
    for pv, labels in train_loader:
        pv = pv.to(device)

        feat = model.encode_image(pv)
        feat = feat / feat.norm(dim=-1, keepdim=True)

        train_feats.append(feat.cpu())
        train_labels.append(labels)

train_feats  = torch.cat(train_feats, dim=0)
train_labels = torch.cat(train_labels, dim=0)

feat_ds = TensorDataset(train_feats, train_labels)
feat_loader = DataLoader(feat_ds, batch_size=256, shuffle=True)


# VALIDATION FEATURES (FIX IMPORTANTE)
val_feats, val_labels = [], []

with torch.no_grad():
    for pv, labels in val_loader:
        pv = pv.to(device)

        feat = model.encode_image(pv)
        feat = feat / feat.norm(dim=-1, keepdim=True)

        val_feats.append(feat.cpu())
        val_labels.append(labels)

val_feats  = torch.cat(val_feats, dim=0)
val_labels = torch.cat(val_labels, dim=0)

val_feat_ds = TensorDataset(val_feats, val_labels)
val_feat_loader = DataLoader(val_feat_ds, batch_size=256, shuffle=False)


print(f"✅ Train features: {train_feats.shape}")
print(f"✅ Val features:   {val_feats.shape}")


# ────────────────────────────────────────────────────────────────
# 4. TEXT PROTOTYPES (SHARED)
# ────────────────────────────────────────────────────────────────

print("\nBuilding text prototypes...")

text_protos = []

with torch.no_grad():
    for i in range(0, 1000, 100):
        j = min(i + 100, 1000)

        prompts = [
            f"a sketch of a {imagenet_class_names[k]}"
            for k in range(i, j)
        ]

        tokens = tokenizer(prompts).to(device)

        t_feat = model.encode_text(tokens)
        t_feat = t_feat / t_feat.norm(dim=-1, keepdim=True)

        text_protos.append(t_feat.cpu())

text_protos = torch.cat(text_protos, dim=0).to(device)

print(f"✅ Text prototypes: {text_protos.shape}")


# ────────────────────────────────────────────────────────────────
# 5. TRAIN FUNCTION + REAL VALIDATION (FIXED)
# ────────────────────────────────────────────────────────────────

def train_adapter(adapter, name, epochs=30, lr=2e-3):

    optimizer = torch.optim.AdamW(adapter.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    scale = model.logit_scale.exp().item()

    print(f"\nTraining {name}...")
    print(f"CLIP logit scale: {scale:.2f}")

    for epoch in range(epochs):

        # ───────── TRAIN ─────────
        adapter.train()
        train_loss = 0.0

        for feats, labels in feat_loader:

            feats, labels = feats.to(device), labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            adapted = adapter(feats)
            adapted = adapted / adapted.norm(dim=-1, keepdim=True)

            logits = scale * adapted @ text_protos.T
            loss = F.cross_entropy(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()


        # ───────── VALIDATION (TRUE SEPARATE SET ✔ FIX) ─────────
        adapter.eval()
        val_loss = 0.0

        with torch.no_grad():
            for feats, labels in val_feat_loader:

                feats, labels = feats.to(device), labels.to(device)

                adapted = adapter(feats)
                adapted = adapted / adapted.norm(dim=-1, keepdim=True)

                logits = scale * adapted @ text_protos.T
                loss = F.cross_entropy(logits, labels)

                val_loss += loss.item()

        scheduler.step()

        if (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch+1:2d}/{epochs} | "
                f"train loss: {train_loss/len(feat_loader):.4f} | "
                f"val loss: {val_loss/len(val_feat_loader):.4f}"
            )

    print(f"✅ {name} training completed")


# ────────────────────────────────────────────────────────────────
# 6. TRAIN ADAPTER 64
# ────────────────────────────────────────────────────────────────

adapter64 = CLIPAdapter(feat_dim=512, bottleneck=64, alpha=0.6).to(device)

print(f"\n✅ Adapter-64 params: {sum(p.numel() for p in adapter64.parameters()):,}")

train_adapter(adapter64, "Adapter-64")


# ────────────────────────────────────────────────────────────────
# 7. TRAIN ADAPTER 128
# ────────────────────────────────────────────────────────────────

adapter128 = CLIPAdapter(feat_dim=512, bottleneck=128, alpha=0.6).to(device)

print(f"\n✅ Adapter-128 params: {sum(p.numel() for p in adapter128.parameters()):,}")

train_adapter(adapter128, "Adapter-128")


Precomputing image features...
✅ Train features: torch.Size([4500, 512])
✅ Val features:   torch.Size([500, 512])

Building text prototypes...
✅ Text prototypes: torch.Size([1000, 512])

✅ Adapter-64 params: 66,113

Training Adapter-64...
CLIP logit scale: 100.00
Epoch  5/30 | train loss: 1.7639 | val loss: 2.2505
Epoch 10/30 | train loss: 1.0442 | val loss: 2.3075
Epoch 15/30 | train loss: 0.6632 | val loss: 2.3524
Epoch 20/30 | train loss: 0.4991 | val loss: 2.4038
Epoch 25/30 | train loss: 0.4401 | val loss: 2.4168
Epoch 30/30 | train loss: 0.4238 | val loss: 2.4190
✅ Adapter-64 training completed

✅ Adapter-128 params: 131,713

Training Adapter-128...
CLIP logit scale: 100.00
Epoch  5/30 | train loss: 1.4611 | val loss: 2.2883
Epoch 10/30 | train loss: 0.5488 | val loss: 2.3266
Epoch 15/30 | train loss: 0.2359 | val loss: 2.4453
Epoch 20/30 | train loss: 0.1533 | val loss: 2.4814
Epoch 25/30 | train loss: 0.1270 | val loss: 2.5070
Epoch 30/30 | train loss: 0.1212 | val loss: 2.510

## 7. Valutazione Post-Fine-Tuning e Report Finale

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SEZIONE 7 — VALUTAZIONE POST FINE-TUNING + PROMPT STUDY (FIXED)
# ═══════════════════════════════════════════════════════════════════

import torch
from tqdm import tqdm

val_loader = DataLoader(val_tensor_ds, batch_size=64, shuffle=False,
                        num_workers=0, pin_memory=True)

# ────────────────────────────────────────────────────────────────
# 1. TEXT FEATURES (CACHE)
# ────────────────────────────────────────────────────────────────

@torch.no_grad()
def build_text_features(clip_model, class_names, prompt_template):

    clip_model.eval()

    feats_all = []

    for start in range(0, len(class_names), 100):
        end = min(start + 100, len(class_names))

        prompts = [
            prompt_template.format(c)
            for c in class_names[start:end]
        ]

        tokens = tokenizer(prompts).to(device)

        feats = clip_model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        feats_all.append(feats.cpu())

    return torch.cat(feats_all, dim=0).to(device)


@torch.no_grad()
def build_text_features_ensemble(clip_model, class_names, prompt_templates):
    """Media le feature testuali su più prompt → prompt ensembling."""
    clip_model.eval()
    all_feats = []

    for template in prompt_templates:
        feats = []
        for start in range(0, len(class_names), 100):
            end     = min(start + 100, len(class_names))
            prompts = [template.format(c) for c in class_names[start:end]]
            tokens  = tokenizer(prompts).to(device)
            f       = clip_model.encode_text(tokens)
            f       = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu())
        all_feats.append(torch.cat(feats, dim=0))

    ensemble = torch.stack(all_feats, dim=0).mean(dim=0)
    ensemble = ensemble / ensemble.norm(dim=-1, keepdim=True)
    return ensemble.to(device)

# ────────────────────────────────────────────────────────────────
# 2. ZERO-SHOT EVALUATION (FIXED — NO ADAPTER)
# ────────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_zero_shot(clip_model, loader, text_feats):

    clip_model.eval()

    scale = clip_model.logit_scale.exp()

    correct, total = 0, 0

    for batch in tqdm(loader, desc="Zero-shot eval", leave=False):

        # FIX: dataset returns (pv, labels)
        pv, labels = batch

        pv = pv.to(device)
        labels = labels.to(device)

        img_feat = clip_model.encode_image(pv)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        logits = scale * img_feat @ text_feats.T
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


# ────────────────────────────────────────────────────────────────
# 3. ADAPTER EVALUATION
# ────────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_adapter(clip_model, adapter_model, loader, text_feats):

    clip_model.eval()
    adapter_model.eval()

    scale = clip_model.logit_scale.exp()

    correct, total = 0, 0

    for batch in tqdm(loader, desc="Adapter eval", leave=False):

        pv, labels = batch  # FIX

        pv = pv.to(device)
        labels = labels.to(device)

        img_feat = clip_model.encode_image(pv)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        adapted = adapter_model(img_feat)
        adapted = adapted / adapted.norm(dim=-1, keepdim=True)

        logits = scale * adapted @ text_feats.T
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


# ═══════════════════════════════════════════════════════════════════
# 4. ZERO-SHOT BASELINE
# ═══════════════════════════════════════════════════════════════════

print("Calcolo Zero-Shot baseline...")

base_prompt = "a sketch of a {}"

text_feats_base = build_text_features(
    model,
    imagenet_class_names,
    base_prompt
)

zeroshot_acc = evaluate_zero_shot(
    model,
    val_loader,
    text_feats_base
)

print(f"\n🎯 Zero-Shot Accuracy: {zeroshot_acc * 100:.2f}%")


# ═══════════════════════════════════════════════════════════════════
# 5. ADAPTER EVALUATION
# ═══════════════════════════════════════════════════════════════════

print("\nValutazione Adapter...")

acc_64 = evaluate_adapter(
    model,
    adapter64,
    val_loader,
    text_feats_base
)

acc_128 = evaluate_adapter(
    model,
    adapter128,
    val_loader,
    text_feats_base
)

delta64 = (acc_64 - zeroshot_acc) * 100
delta128 = (acc_128 - zeroshot_acc) * 100


# ═══════════════════════════════════════════════════════════════════
# 6. PROMPT STUDY (BEST MODEL = 128)
# ═══════════════════════════════════════════════════════════════════

print("\nTesting different prompts...\n")

prompt_list = [
    "a sketch of a {}",
    "a black and white sketch of a {}",
    "a hand-drawn sketch of a {}",
    "a pencil drawing of a {}",
    "a line drawing of a {}"
]

best_model = adapter128
prompt_results = []

for p in prompt_list:

    text_feats = build_text_features(
        model,
        imagenet_class_names,
        p
    )

    acc = evaluate_adapter(
        model,
        best_model,
        val_loader,
        text_feats
    )

    prompt_results.append((p, acc))


# ═══════════════════════════════════════════════════════════════════
# 7. FINAL REPORT
# ═══════════════════════════════════════════════════════════════════

# Prompt Ensembling
sketch_prompts = [
    "a sketch of a {}",
    "a black and white sketch of a {}",
    "a hand-drawn sketch of a {}",
    "a pencil drawing of a {}",
    "a line drawing of a {}"
]

text_feats_ensemble = build_text_features_ensemble(
    model, imagenet_class_names, sketch_prompts
)

acc_zs_ensemble  = evaluate_zero_shot(model, val_loader, text_feats_ensemble)

acc_64_ensemble = evaluate_adapter(model, adapter64, val_loader, text_feats_ensemble)
delta_64_ensemble = (acc_64_ensemble - zeroshot_acc) * 100
acc_128_ensemble = evaluate_adapter(model, adapter128, val_loader, text_feats_ensemble)
delta_128_ensemble = (acc_128_ensemble - zeroshot_acc) * 100

print("\n" + "="*75)
print("        FINAL REPORT — CLIP ADAPTER STUDY")
print("="*75)
print(f"  Dataset:   ImageNet-Sketch (domain shift)")
print(f"  Backbone:  CLIP ViT-B/16 — completamente frozen")
print(f"  PEFT:      CLIP-Adapter (bottleneck MLP + residual blending)")
print("="*75)

print(f"\n{'Method':<40}{'Accuracy':>10}{'Gain':>10}")
print("-"*75)
print(f"{'Zero-shot CLIP (base prompt)':<40}{zeroshot_acc*100:>9.2f}%{'—':>10}")
print(f"{'Zero-shot CLIP (ensemble 5 prompts)':<40}{acc_zs_ensemble*100:>9.2f}%{(acc_zs_ensemble-zeroshot_acc)*100:>+9.2f}p")
print(f"{'CLIP-Adapter bottleneck=64':<40}{acc_64*100:>9.2f}%{delta64:>+9.2f}p")
print(f"{'CLIP-Adapter b=64 + Ensemble':<40}{acc_64_ensemble*100:>9.2f}%{delta_64_ensemble:>+9.2f}p")
print(f"{'CLIP-Adapter bottleneck=128':<40}{acc_128*100:>9.2f}%{delta128:>+9.2f}p")
print(f"{'CLIP-Adapter b=128 + Ensemble':<40}{acc_128_ensemble*100:>9.2f}%{delta_128_ensemble:>+9.2f}p")
print("="*75)

print(f"\nAlpha imparato Adapter-64:  {torch.sigmoid(adapter64.alpha).item():.4f}")
print(f"Alpha imparato Adapter-128: {torch.sigmoid(adapter128.alpha).item():.4f}")

print("\nPrompt comparison (Adapter-128):\n")
best_prompt, best_acc = None, 0
for p, acc in prompt_results:
    print(f"  {p:<45} {acc*100:.2f}%")
    if acc > best_acc:
        best_acc, best_prompt = acc, p
print("-"*75)
print(f"  Best single prompt: {best_prompt}")
print(f"  Best single prompt accuracy: {best_acc*100:.2f}%")
print("="*75)

Calcolo Zero-Shot baseline...



🎯 Zero-Shot Accuracy: 42.64%

Valutazione Adapter...



Testing different prompts...




        FINAL REPORT — CLIP ADAPTER STUDY
  Dataset:   ImageNet-Sketch (domain shift)
  Backbone:  CLIP ViT-B/16 — completamente frozen
  PEFT:      CLIP-Adapter (bottleneck MLP + residual blending)

Method                                    Accuracy      Gain
---------------------------------------------------------------------------
Zero-shot CLIP (base prompt)                42.64%         —
Zero-shot CLIP (ensemble 5 prompts)         43.57%    +0.93p
CLIP-Adapter bottleneck=64                  48.17%    +5.53p
CLIP-Adapter bottleneck=128                 49.46%    +6.82p
CLIP-Adapter b=128 + Ensemble               49.62%    +6.98p

Alpha imparato Adapter-64:  0.7595
Alpha imparato Adapter-128: 0.7503

Prompt comparison (Adapter-128):

  a sketch of a {}                              49.46%
  a black and white sketch of a {}              48.08%
  a hand-drawn sketch of a {}                   48.98%
  a pencil drawing of a {}                      48.67%
  a line drawing of a {}       